In [ ]:
import pandas as pd
import glob
import os
import re

# ==============================
# 1. READ ALL CSV FILES
# ==============================

DATA_FOLDER = "../data_raw/"
OUTPUT_FOLDER = "../data_processed/"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

files = glob.glob(os.path.join(DATA_FOLDER, "*.csv"))

print("Number of CSV files:", len(files))

In [ ]:
def get_file_number(path):
    name = os.path.basename(path)

    # Base file: ...-001.csv -> 0
    # ...-001-1.csv -> 1
    # ...-001-17.csv -> 17

    match = re.search(r'-001(?:-(\d+))?\.csv$', name)

    if match:
        return int(match.group(1)) if match.group(1) else 0

    return 999


files = sorted(files, key=get_file_number)

for i, f in enumerate(files, start=1):
    print(i, os.path.basename(f))

In [ ]:
COURSE_CODES = {
    0: "ADY201M",
    1: "PCP291",
    2: "PRJ301",
    3: "PRF193",
    4: "PFP191",
    5: "PRJ321",
    6: "DAP391M",
    7: "CSD201",
    8: "IOT101",
    9: "CSD203",
    10: "PRO192",
    11: "IOT102",
    12: "PRJ30X",
    13: "LAB211",
    14: "OSG202",
    15: "CEA201",
    16: "DBI202",
    17: "PRF192",
}

course_map = []

for f in files:

    file_number = get_file_number(f)

    course_id = COURSE_CODES[file_number]

    course_map.append({
        "course_proxy": f"C{file_number + 1:02d}",
        "course_id": course_id,
        "source_file": os.path.basename(f)
    })

course_map_df = pd.DataFrame(course_map)

course_map_df.to_csv(
    os.path.join(OUTPUT_FOLDER, "course_map.csv"),
    index=False,
    encoding="utf-8-sig"
)

course_map_df

In [ ]:
all_data = []

for f in files:

    file_number = get_file_number(f)
    course_id = COURSE_CODES[file_number]

    df = pd.read_csv(
        f,
        dtype=str,
        encoding="utf-8-sig"
    )

    # Chuẩn hóa tên cột
    df.columns = [col.strip() for col in df.columns]

    # Các column bắt buộc
    required_columns = [
        "Mã học kỳ",
        "Mã lớp",
        "NO.",
        "CODE",
        "TOTAL",
        "STATUS"
    ]

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        print(course_id, "Missing:", missing)
        continue

    # Giữ row number gốc để truy vết
    df["source_row"] = range(2, len(df) + 2)

    # Chỉ giữ các dòng sinh viên thật.
    # Các dòng footer của website sẽ không có NO. dạng số.
    df["NO_numeric"] = pd.to_numeric(
        df["NO."],
        errors="coerce"
    )

    df = df[df["NO_numeric"].notna()].copy()

    # Chỉ giữ các semester hợp lệ
    semester_pattern = r"^(SP|SU|FA)\d{4}$"

    df = df[
        df["Mã học kỳ"]
        .astype(str)
        .str.match(semester_pattern, na=False)
    ].copy()

    # Thêm thông tin nguồn
    df["course_id"] = course_id
    df["source_file"] = os.path.basename(f)

    # Chỉ giữ schema chung
    df = df[
        [
            "course_id",
            "source_file",
            "source_row",
            "Mã học kỳ",
            "Mã lớp",
            "NO.",
            "CODE",
            "TOTAL",
            "STATUS"
        ]
    ]

    df = df.rename(
        columns={
            "Mã học kỳ": "semester",
            "Mã lớp": "class_id",
            "NO.": "source_no",
            "CODE": "student_id",
            "TOTAL": "total_score",
            "STATUS": "status_raw"
        }
    )

    all_data.append(df)

master = pd.concat(
    all_data,
    ignore_index=True
)

print(master.shape)

master.head()

In [ ]:
master["student_id"] = (
    master["student_id"]
    .astype(str)
    .str.strip()
)

master["status_raw"] = (
    master["status_raw"]
    .astype(str)
    .str.strip()
)

master["semester"] = (
    master["semester"]
    .astype(str)
    .str.strip()
)

master["total_score"] = pd.to_numeric(
    master["total_score"],
    errors="coerce"
)

In [ ]:
season_order = {
    "SP": 0,
    "SU": 1,
    "FA": 2
}


def semester_to_order(value):

    match = re.match(
        r"^(SP|SU|FA)(\d{4})$",
        str(value)
    )

    if not match:
        return None

    season = match.group(1)
    year = int(match.group(2))

    return year * 3 + season_order[season]


master["semester_order"] = (
    master["semester"]
    .apply(semester_to_order)
)

In [ ]:
duplicate_key = [
    "course_id",
    "student_id",
    "semester"
]

master["duplicate_count"] = (
    master
    .groupby(duplicate_key)["student_id"]
    .transform("size")
)

master["is_duplicate_candidate"] = (
    master["duplicate_count"] > 1
)

In [ ]:
duplicates = master[
    master["is_duplicate_candidate"]
].copy()

duplicates = duplicates.sort_values(
    [
        "course_id",
        "student_id",
        "semester",
        "source_row"
    ]
)

duplicates.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "duplicate_candidates.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print("Duplicate candidate rows:",
      len(duplicates))

In [ ]:
master.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "master_raw.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
master_sql = (
    master
    .sort_values(
        [
            "course_id",
            "student_id",
            "semester_order",
            "source_row"
        ]
    )
    .drop_duplicates(
        subset=[
            "course_id",
            "student_id",
            "semester"
        ],
        keep="last"
    )
    .copy()
)

In [ ]:
master_sql = master_sql.sort_values(
    [
        "course_id",
        "student_id",
        "semester_order"
    ]
)

master_sql["attempt_no"] = (
    master_sql
    .groupby(
        [
            "course_id",
            "student_id"
        ]
    )
    .cumcount()
    + 1
)

In [ ]:
master_sql.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "master_sql_ready.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
print(
    "Raw valid rows:",
    len(master)
)

print(
    "Candidate deduplicated rows:",
    len(master_sql)
)

print(
    "Unique students:",
    master_sql["student_id"].nunique()
)

print(
    "Courses:",
    master_sql["course_id"].nunique()
)

print(
    "Semesters:",
    master_sql["semester"].nunique()
)

print(
    "Semester range:",
    master_sql["semester_order"].min(),
    "→",
    master_sql["semester_order"].max()
)

In [ ]:
trajectory = (
    master_sql
    .groupby(
        [
            "student_id",
            "course_id"
        ]
    )
    .size()
    .reset_index(name="attempt_count")
)

print(
    "Student-Course trajectories:",
    len(trajectory)
)

print(
    ">= 2 attempts:",
    (trajectory["attempt_count"] >= 2).sum()
)

print(
    ">= 3 attempts:",
    (trajectory["attempt_count"] >= 3).sum()
)

print(
    ">= 4 attempts:",
    (trajectory["attempt_count"] >= 4).sum()
)

In [ ]:
# ==========================================
# TV1 WEEK 2 - QA SUMMARY
# ==========================================

duplicate_key = [
    "course_id",
    "student_id",
    "semester"
]

extra_duplicate_rows = (
    master
    .duplicated(
        subset=duplicate_key,
        keep="first"
    )
    .sum()
)

trajectory = (
    master_sql
    .groupby(
        [
            "student_id",
            "course_id"
        ]
    )
    .size()
    .reset_index(
        name="attempt_count"
    )
)

semester_table = (
    master_sql[
        [
            "semester",
            "semester_order"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "semester_order"
    )
)

print(
    "===== TV1 WEEK 2 QA ====="
)

print(
    "Valid rows before dedup:",
    len(master)
)

print(
    "Extra duplicate rows:",
    extra_duplicate_rows
)

print(
    "Rows after dedup:",
    len(master_sql)
)

print(
    "Unique students:",
    master_sql[
        "student_id"
    ].nunique()
)

print(
    "Course proxies:",
    master_sql[
        "course_id"
    ].nunique()
)

print(
    "Semester range:",
    semester_table.iloc[0][
        "semester"
    ],
    "->",
    semester_table.iloc[-1][
        "semester"
    ]
)

print(
    "Unique semesters:",
    master_sql[
        "semester"
    ].nunique()
)

print(
    "Student-Course trajectories:",
    len(trajectory)
)

print(
    ">=2 attempts:",
    (
        trajectory[
            "attempt_count"
        ] >= 2
    ).sum()
)

print(
    ">=3 attempts:",
    (
        trajectory[
            "attempt_count"
        ] >= 3
    ).sum()
)

print(
    ">=4 attempts:",
    (
        trajectory[
            "attempt_count"
        ] >= 4
    ).sum()
)

In [ ]:
for n in range(1, 8):

    count = (
        trajectory["attempt_count"]
        >= n
    ).sum()

    print(
        f">={n} attempts:",
        count
    )